## Data Download

In [ ]:
from src.dataset_downloader import DatasetDownloader

DatasetDownloader(dataset='DAIC_WoZ')
DatasetDownloader(dataset='Androids_Corpus')

## Data Inspection

### DAIC-WoZ

In [4]:
from src.dataset_loader import DAICWoZDataset
from IPython.display import Audio

PARTICIPANT_IDX = 0
SEGMENT_IDX = 0

dataset = DAICWoZDataset(train_or_dev='train')
X_audio, X_text, y = dataset[PARTICIPANT_IDX]
print(X_text[SEGMENT_IDX])
Audio(X_audio[SEGMENT_IDX], rate=dataset.SAMPLE_RATE)

okay how 'bout yourself


### Androids-Corpus

In [2]:
from src.dataset_loader import AndroidsCorpusDataset
from IPython.display import Audio

FOLD_IDX = 0
PARTICIPANT_IDX = 0
SEGMENT_IDX = 0

dataset = AndroidsCorpusDataset(fold=FOLD_IDX, train_or_test='train')
X_audio, X_text, y = dataset[PARTICIPANT_IDX]
print(X_text[SEGMENT_IDX])
Audio(X_audio[SEGMENT_IDX], rate=dataset.SAMPLE_RATE)

 state pochi in campagna, state aiutate un po' la mia moglie, insomma in casa, poi il lobby, e vedere il calcio. Mi piace molto il calcio. Questo è seruto. E poi parlato con gli amici in piazza, in piazzetta nostra, Poi questo debbo dire più.


## Experiments

In [1]:
from src.dataset_loader import DAICWoZDataset, AndroidsCorpusDataset
from src.feature_extractor import AudioFeatureExtractor, TextFeatureExtractor
from src.model import MultimodalClassifier
import torch
from torchinfo import summary
import os

device = (
    'cuda:0' if torch.cuda.is_available() else
    'mps:0' if torch.backends.mps.is_available() else
    'cpu'
)

### DAIC-WoZ

In [2]:
dataset = DAICWoZDataset(train_or_dev='train')
audio_vectorizer = AudioFeatureExtractor(model_name='Wav2Vec2', device=device)
text_vectorizer = TextFeatureExtractor(model_name='BERT', device=device)

audio_features = audio_vectorizer(dataset.X_audio[0])
text_features = text_vectorizer(dataset.X_text[0])

audio_features.shape, text_features.shape

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(torch.Size([1, 34449, 768]), torch.Size([1, 2384, 768]))

### Androids-Corpus

In [4]:
fold = 0
dataset = AndroidsCorpusDataset(fold=fold, train_or_test='train')
audio_vectorizer = AudioFeatureExtractor(model_name='HuBERT', device=device)
text_vectorizer = TextFeatureExtractor(model_name='ItalianBERT', device=device)

audio_features = audio_vectorizer(dataset.X_audio[0])
text_features = text_vectorizer(dataset.X_text[0])

audio_features.shape, text_features.shape

(torch.Size([1, 8047, 1024]), torch.Size([1, 428, 768]))

In [ ]:
# for fold in range(5):
#     dataset = AndroidsCorpusDataset(fold=fold, train_or_test='train')

## Training

In [5]:
from src.dataset_loader import AndroidsCorpusDataset
from src.feature_extractor import AudioFeatureExtractor, TextFeatureExtractor
from src.model import MultimodalClassifier
from torch.utils.data import DataLoader
import torch

fold = 0
batch_size = 6
device = (
    'cuda:0' if torch.cuda.is_available() else
    'mps:0' if torch.backends.mps.is_available() else
    'cpu'
)
dataset = AndroidsCorpusDataset(fold=fold, train_or_test='train')
model = MultimodalClassifier(
    audio_vectorizer=AudioFeatureExtractor(model_name='HuBERT', device=device),
    text_vectorizer=TextFeatureExtractor(model_name='ItalianBERT', device=device),
).to(device)


In [6]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=dataset.collate_fn)

for batch_idx, (X_audio, X_text, y) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}:")
    # print(len(X_audio), len(X_text), len(y))

Batch 1:
Batch 2:
Batch 3:
Batch 4:
Batch 5:
Batch 6:
Batch 7:
Batch 8:
Batch 9:
Batch 10:
Batch 11:
Batch 12:
Batch 13:
Batch 14:
Batch 15:
Batch 16:


In [11]:
# with torch.no_grad():
#     output = model(audio_segments, text_segments).item()
# output

0.49531444907188416

In [9]:
# import librosa
# import os

# Example usage
# path = os.path.join(os.getcwd(), 'data/raw/DAIC_WoZ/301_AUDIO.wav')
# waveform, _ = librosa.load(path, sr=16000)

torch.Size([41167, 768])